In [ ]:
import os
import pathlib

import plotly.express
import plotly.graph_objects
import numpy as np
import seaborn
import pandas as pd

def load_df(file_path: pathlib.Path | str) -> pd.DataFrame:
    file_path = pathlib.Path(file_path)
    if file_path.suffix == ".csv":
        return pd.read_csv(file_path)
    if file_path.suffix == ".json":
        return pd.read_json(file_path)
    if file_path.suffix == ".jsonl":
        return pd.read_json(file_path, lines=True)
    else:
        raise ValueError(f"Unsupported file format: {file_path.suffix}")

In [3]:
def extract_answer(response: str) -> str:
    """Extract the final answer from the model response."""
    for line in reversed(response.splitlines()):
        line = line.strip()
        if line.upper().startswith("ANSWER:"):
            return line.split(":", 1)[1].strip()
    # Fallback: return last non-empty line
    for line in reversed(response.splitlines()):
        if line.strip():
            return line.strip()
    return ""


def normalize(text: str) -> str:
    """Normalize an answer string for comparison."""
    t = str(text).strip().lower().rstrip(".")
    for ch in ("$", ",", "%"):
        t = t.replace(ch, "")
    try:
        return str(float(t))
    except ValueError:
        return t


def answers_match(expected: str, predicted: str) -> bool:
    """Check whether the predicted answer matches the expected one."""
    return normalize(expected) == normalize(predicted)


In [ ]:
df = []
for file in pathlib.Path("../predictions/").glob("*.jsonl"):
    f = load_df(file)
    if file.stem.count("=") == 2:
        f["augmentation"] = file.stem.split("___")[1].split("=")[0]
    else:
        f["augmentation"] = "base"
    df.append(f)
df_all = pd.concat(df, ignore_index=True)
df_all["is_correct"] = df_all.apply(lambda row: answers_match(row["answer"], extract_answer(row["predicted_result"])), axis=1)

In [ ]:
pathlib.Path("plots").mkdir(exist_ok=True)

for augmentation in set(df_all["augmentation"].unique()) - {"base"}:
    for model in df_all["prediction_api_model"].unique():
        df = df_all
        df = df[df["prediction_api_model"] == model]
        df = df[(df["augmentation"] == augmentation) | (df["augmentation"] == "base")]

        base = df[df["augmentation"] == "base"]
        augmented_same_model = df[(df["augmentation"] == augmentation) & (df["question_api_model"] == model)]
        augmented_different_model = df[(df["augmentation"] == augmentation) & (df["question_api_model"] != model)]
        
        base_accs = base.groupby("id")["is_correct"].mean()
        same_model_accs = augmented_same_model.groupby("id")["is_correct"].mean()
        different_model_accs = augmented_different_model.groupby("id")["is_correct"].mean()
        
        assert (base_accs.index == same_model_accs.index).all() and (base_accs.index == different_model_accs.index).all(), "Mismatched question IDs between base and augmented data"
        fig = plotly.express.bar(
            title=f"Model: {model}, Augmentation: {augmentation}",
            data_frame=pd.DataFrame({
                "Base": base_accs,
                "Augmented (Same Model)": same_model_accs,
                "Augmented (Diff Model)": different_model_accs
            }),
                labels={"value": "Accuracy", "index": "Question ID"},
                barmode="group"
        )
        fig.show()
        fig.write_image(f"plots/{model}_{augmentation}.png")
        